# Musanga NB03 — V4.009 | Complete overlapping inference & validation (NO TRAINING)

A **standalone inference-first notebook**, not a patch on the RGB training notebook. All three verified **V4.007** ResNet18 U-Net checkpoints are **read-only**. All **76/78 RGB-evaluable field plots** and **34 exhaustive circles** are evaluated with the **same 256-pixel native resolution / 128-pixel global-grid stride / Hann-weighted probability blending**. The evaluation saves per-plot and per-circle metrics, model predictions vs reference scatterplots, and detailed probability/RGB/reference/error panels for MOS **3, 9, 29, 26**.

**Execute every code block in sequence. There is no training cell.** Final outputs: `Final/Outputs/v4/03_overlap_evaluation_v4_009/`. Historical V4.007 results are preserved separately for non-overlapping comparison.

Note: inference blending reduces grid-edge artefacts but cannot, by itself, correct missing species information. Further predictor inventory at the notebook end is read-only.


## 00 — Load established NB00/NB01 reference contract and V4.007 checkpoints

In [ ]:
# 00 — Environment, frozen inputs and run settings
%pip -q install geopandas pyogrio shapely rasterio scipy scikit-learn torch matplotlib segmentation-models-pytorch
from pathlib import Path
import json, math, random, copy, time, warnings, os
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
from shapely.ops import unary_union
from shapely import make_valid
from shapely.strtree import STRtree
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
DRIVE=Path('/content/drive/MyDrive')
ROOT=DRIVE/'Colab Notebooks/Musanga/Final'
assert ROOT.is_dir(), ROOT
DATA=ROOT/'Data'
REF_DIR=ROOT/'Outputs/v4/01_reference_field78_combined'
CONTRACT=json.loads((REF_DIR/'01_field78_reference_contract.json').read_text())
assert CONTRACT['schema']=='musanga_v4_field78_combined_reference_final_2',CONTRACT['schema']
NB00=json.loads(Path(CONTRACT['input_manifest']).read_text())
assert NB00['n_field_plots']==78
SPLITS=pd.read_csv(CONTRACT['cv_manifest'])
assert len(SPLITS)==9096,(len(SPLITS),'Expected 3032 unique windows × 3 folds; changed source audit?')
K=['cv_fold','mosaic','row','col','width','height']
assert not SPLITS.duplicated(K).any()
assert SPLITS.groupby('cv_fold').size().eq(3032).all()
assert set(SPLITS.split).issubset({'train','val','excluded_spatial_guard','excluded_no_validation_pixels'})
PLOTS=gpd.read_file(NB00['field_support_qa']).to_crs('EPSG:32635')
PLOT_FOLDS=pd.read_csv(CONTRACT['folds_csv'])
assert len(PLOTS)==len(PLOT_FOLDS)==78 and PLOT_FOLDS.source_id.is_unique
PLOTS=PLOTS.merge(PLOT_FOLDS[['source_id','site_key','cv_fold']],left_on='Plot_name',right_on='source_id',validate='one_to_one')
CIRCLE_FOLDS=pd.read_csv(REF_DIR/'01_field78_circle_folds.csv')
# Adjustable, deterministic experiment settings
SMOKE_TEST=False           # FIXED FULL RUN ONLY; no smoke option
RUN_FOLDS=[0,1,2]         # FIXED all three spatial folds
SEED=20261113
CHIP=256                   # 256 native pixels; no spatial downsampling
BATCH_SIZE=8
EPOCHS=25
STEPS_PER_EPOCH=100
EVAL_BATCH_SIZE=8  # batched exhaustive held-out inference, mosaic handle opened once
LEARNING_RATE=2e-4
PATIENCE=4
POSITIVE_SAMPLE_BOOST=2.5
MAX_VAL_WINDOWS_PER_SITE=35  # early-stopping cache only; FINAL evaluation is uncapped
MAX_TRAIN_WINDOWS_PER_SITE=180
NUM_WORKERS=0              # Google Drive + GDAL: safest in Colab
OUT=ROOT/'Outputs/v4/02_rgb_unet_field78_v4_007_full'
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type=='cuda', 'Select a GPU runtime for inference'
print('Device:',DEVICE,'| V4.009 INFERENCE ONLY; V4.007 checkpoints')
print('Final manifest:',CONTRACT['cv_manifest'])
print('Observed window splits:');display(SPLITS.groupby(['cv_fold','site','split']).size().unstack(fill_value=0))

# Canonical immutable numbered experiment; no shared output with earlier experiments.
VERSION='V4.007'
assert SMOKE_TEST is False and RUN_FOLDS==[0,1,2]
assert EPOCHS==25 and STEPS_PER_EPOCH==100
assert MAX_TRAIN_WINDOWS_PER_SITE==180 and MAX_VAL_WINDOWS_PER_SITE==35
assert OUT.name=='02_rgb_unet_field78_v4_007_full',OUT
CHECKPOINTS={fold:OUT/f'NB02_V4_007_fold{fold}_best.pt' for fold in RUN_FOLDS}
print('STRICT FULL MODEL RUN:',VERSION,'\nOUTPUT:',OUT)
print('FOLD CHECKPOINTS:',{f:str(p) for f,p in CHECKPOINTS.items()})

N_BINS=512 # used by secondary complete-circle metrics if plot evaluation is skipped

INNER_BLOCK_PX=2048
INNER_GUARD_PX=256
INNER_VAL_FRACTION=.20
SOURCE_CODES={0:'unknown',1:'partial_crown',2:'exhaustive_circle',3:'exhaustive_plot'}

# V4.007 reuses ONLY V4.003 source-labelled chip cache bytes when exact
# manifest/input fingerprint matches; no model weights are transferred.
READ_ONLY_CACHE_V003=ROOT/'Outputs/v4/02_rgb_unet_field78_v4_003_full/chip_cache'
INNER_SELECTION_SOURCES=(2,3)   # exhaustive circles and plots ONLY
TRAINING_SOURCES=(1,2,3)         # individual positive/negative crowns training ONLY
assert VERSION=='V4.007' and OUT.name=='02_rgb_unet_field78_v4_007_full'

import gc,resource
def memory_snapshot(tag):
    print('RAM diagnostic',tag,'max_RSS_GB',round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024**2,2),flush=True)


## 01 — Reference geometries, orthomosaics, masks and model architecture

In [ ]:
# 01 — Geometry references and fast spatial indexes
CROWNS=gpd.read_file(NB00['exhaustive_plot_crowns']).to_crs('EPSG:32635')
CIRCLE_ROI=gpd.read_file(NB00['circle_complete_roi']).to_crs('EPSG:32635')
CIRCLE_POS=gpd.read_file(NB00['circle_positive_crowns']).to_crs('EPSG:32635')
assert len(CROWNS)==3324 and len(CIRCLE_ROI)==34
assert 'Class_ID' in CROWNS
def polygons(g):
    if g is None or g.is_empty:return []
    if g.geom_type=='Polygon':return [g]
    if g.geom_type=='MultiPolygon':return list(g.geoms)
    return [a for v in getattr(g,'geoms',[]) for a in polygons(v)]
def fix(g):
    if g is None or g.is_empty:return None
    if g.is_valid:return g
    q=unary_union(polygons(make_valid(g)))
    if q.is_empty or not q.is_valid:return None
    if abs(q.area-g.area)/max(q.area,g.area,1e-12)>.05:return None
    return q
CROWNS.geometry=[fix(g) for g in CROWNS.geometry]
assert CROWNS.geometry.notna().all(),'Unrepaired plot crowns; see NB00 QA'
CIRCLE_ROI.geometry=[fix(g) for g in CIRCLE_ROI.geometry]
CIRCLE_POS.geometry=[fix(g) for g in CIRCLE_POS.geometry]
assert CIRCLE_ROI.geometry.notna().all()
CIRCLE_ROI['circle_id']=np.arange(len(CIRCLE_ROI))
CIRCLE_ROI=CIRCLE_ROI.merge(CIRCLE_FOLDS[['circle_id','site_key','cv_fold',
    'cross_fold_plot_intersection']],on='circle_id',validate='one_to_one')
CROWN_POS=list(CROWNS.loc[pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CROWN_NEG=list(CROWNS.loc[~pd.to_numeric(CROWNS.Class_ID).eq(1)].geometry)
CIRCLE_POS_GEOMS=[g for g in CIRCLE_POS.geometry if g is not None and not g.is_empty]
# Partial site references from old V3_1 audit, same positive=1 / other known class negative.
PARTIAL={}
for site,path in NB00['partial_site_crowns'].items():
    d=gpd.read_file(path).to_crs('EPSG:32635')
    d.geometry=[fix(g) for g in d.geometry]
    d=d[d.geometry.notna() & pd.to_numeric(d.Class_ID,errors='coerce').notna()]
    PARTIAL[site]=d
P_GEOMS=list(PLOTS.geometry)
C_GEOMS=list(CIRCLE_ROI.geometry)
P_T=STRtree(P_GEOMS); C_T=STRtree(C_GEOMS)
CP_T=STRtree(CROWN_POS);CN_T=STRtree(CROWN_NEG)
KPOS_T=STRtree(CIRCLE_POS_GEOMS)
PARTIAL_T={}
for site,d in PARTIAL.items():
    pos=list(d.loc[pd.to_numeric(d.Class_ID).eq(1)].geometry)
    neg=list(d.loc[~pd.to_numeric(d.Class_ID).eq(1)].geometry)
    PARTIAL_T[site]=(pos,neg,STRtree(pos) if pos else None,
                     STRtree(neg) if neg else None)
RGB={}
for item in sorted(SPLITS.mosaic.unique()):
    q=DATA/'Ortho_RGB'/item
    assert q.exists(),q
    with rasterio.open(q) as ds:
        assert ds.crs.to_epsg()==32635,(item,ds.crs)
        assert ds.count>=3,(item,ds.count)
        RGB[item]=dict(path=str(q),transform=ds.transform,site=next(
            iter(SPLITS.loc[SPLITS.mosaic==item,'site'].unique())))
# NOTE: ROI.shp is the authoritative six-mosaic modelling ROI (not the old
# circle-only support). The validation/labelled field plot is also required.
ROIS=gpd.read_file(DATA/'Ortho_RGB/ROI.shp').to_crs('EPSG:32635')
ROI_GEOMS=list(ROIS.geometry)
ROI_T=STRtree(ROI_GEOMS)
print('Loaded',len(PLOTS),'plots,',len(CIRCLE_ROI),'circles,',
      len(CROWN_POS),'Musanga plot crowns and',len(CIRCLE_POS_GEOMS),'circle crowns')


In [ ]:
# 02 — Rasterize strict plot > circle > partial supervision on demand
# Geometry-level spatial queries restrict operations to the local 256px footprint.
def geom_hits(tree,geoms,ext):
    if tree is None:return []
    return [geoms[int(i)] for i in tree.query(ext,predicate='intersects')]
_ACTIVE_RASTER=None
_ACTIVE_NAME=None
def rgb_window(raster_path,win):
    from contextlib import nullcontext
    name=Path(raster_path).name
    ds_context=nullcontext(_ACTIVE_RASTER) if (_ACTIVE_RASTER is not None and _ACTIVE_NAME==name) else rasterio.open(raster_path)
    with ds_context as ds:
        a=ds.read([1,2,3],window=win,masked=True)
        valid=ds.dataset_mask(window=win)>0
        valid &= ~np.ma.getmaskarray(a).any(axis=0)
        x=np.asarray(a.filled(0),dtype=np.float32)
        # RGB products expected standard 8-bit; prevent silent incorrect scaling.
        if np.issubdtype(np.dtype(ds.dtypes[0]),np.integer) and np.dtype(ds.dtypes[0]).itemsize==1:
            x=x/255.
        else:
            raise RuntimeError(f'Non-uint8 RGB {ds.dtypes}: {raster_path}; define radiometric scaling first')
    return x,np.asarray(valid,bool)
def supervision(mosaic,win,fold,role,with_source=False):
    site=RGB[mosaic]['site'];t=RGB[mosaic]['transform']
    h,w=int(win.height),int(win.width)
    affine=rasterio.windows.transform(win,t)
    footprint=box(*rasterio.windows.bounds(win,t))
    def burn(gs):
        return rasterize(((g,1) for g in gs if g is not None and not g.is_empty),
            out_shape=(h,w),transform=affine,fill=0,dtype='uint8').astype(bool) if gs else np.zeros((h,w),bool)
    roi=burn(geom_hits(ROI_T,ROI_GEOMS,footprint))
    pids=[int(i) for i in P_T.query(footprint,predicate='intersects')
          if PLOTS.iloc[int(i)].site_key==site and
          (role=='train' and PLOTS.iloc[int(i)].cv_fold!=fold or
           role=='val' and PLOTS.iloc[int(i)].cv_fold==fold)]
    plot=burn([P_GEOMS[i] for i in pids])
    # Every other fold's field plot is out of scope for val; all exhaustive
    # plot geometries (not only selected) override circle labels.
    allplot=burn([P_GEOMS[int(i)] for i in P_T.query(footprint,predicate='intersects')
                  if PLOTS.iloc[int(i)].site_key==site])
    cids=[int(i) for i in C_T.query(footprint,predicate='intersects')
          if CIRCLE_ROI.iloc[int(i)].site_key==site and
          not CIRCLE_ROI.iloc[int(i)].cross_fold_plot_intersection and
          (role=='train' and CIRCLE_ROI.iloc[int(i)].cv_fold!=fold)]
    circle=burn([C_GEOMS[i] for i in cids])&~allplot if role=='train' else np.zeros((h,w),bool)
    # Partial crowns must NEVER relabel any exhaustive circle, including
    # an outer-held-out circle. Train circles retain exclusive authority.
    allcircle=burn([C_GEOMS[int(i)] for i in C_T.query(footprint,predicate='intersects')
        if CIRCLE_ROI.iloc[int(i)].site_key==site])
    posplot=burn(geom_hits(CP_T,CROWN_POS,footprint))&plot
    negplot=burn(geom_hits(CN_T,CROWN_NEG,footprint))&plot
    poscircle=burn(geom_hits(KPOS_T,CIRCLE_POS_GEOMS,footprint))&circle
    part_pos=np.zeros((h,w),bool);part_neg=np.zeros((h,w),bool)
    if role=='train':
        positives,negatives,tp,tn=PARTIAL_T[site]
        part_pos=burn(geom_hits(tp,positives,footprint))&~allplot&~allcircle
        part_neg=burn(geom_hits(tn,negatives,footprint))&~allplot&~allcircle
    source=np.zeros((h,w),dtype=np.uint8)
    source[part_pos|part_neg]=1
    source[circle]=2
    source[plot]=3
    target=(posplot|poscircle|part_pos)
    conflicting=(posplot&negplot)|(part_pos&part_neg)
    weight=roi&(plot|circle|part_pos|part_neg)&~conflicting
    source[~weight.astype(bool)]=0
    assert not np.any((source==0)&(weight>0))
    return (target.astype(np.float32),weight.astype(np.float32),plot.astype(bool),source) if with_source else (target.astype(np.float32),weight.astype(np.float32),plot.astype(bool))
def get_chip(mosaic,win,fold,role):
    x,rgbvalid=rgb_window(RGB[mosaic]['path'],win)
    y,w,plot,source=supervision(mosaic,win,fold,role,with_source=True)
    w*=rgbvalid.astype(np.float32)
    source[~rgbvalid]=0
    return x,y,w,plot,source
print('Prepared lazy RGB + spatially strict masks. No imagery loaded globally.')


In [ ]:
# 05 — Restore the working V3_1 ResNet18 U-Net + masked BCE, AMP in training
import segmentation_models_pytorch as smp
import torch.nn.functional as F
class ResNet18UNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=smp.Unet(encoder_name='resnet18',encoder_weights='imagenet',
            in_channels=3,classes=1)
    def forward(self,x):return self.net(x)  # B,1,H,W
def masked_loss(logits,y,weight):
    """Original V3_1 masked pixelwise BCE; unknown/unlabelled pixels have weight=0."""
    bce=F.binary_cross_entropy_with_logits(logits.float(),y.float(),reduction='none')
    return (bce*weight).sum()/weight.sum().clamp_min(1.)

AMP=DEVICE.type=='cuda'
def run_epoch(model,loader,optimizer=None,scaler=None):
    model.train(optimizer is not None)
    totals=[];site_losses={}
    for x,y,w,site,src in loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        w=w.to(DEVICE,non_blocking=True)
        src=src.to(DEVICE,non_blocking=True)
        if not w.any():continue
        with torch.set_grad_enabled(optimizer is not None):
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                logits=model(x)
            if optimizer is None:
                assert not torch.any((src==1)&(w>0)), 'PARTIAL CROWNS IN MODEL SELECTION'
                assert not torch.any((src==0)&(w>0)), 'UNKNOWN PIXELS IN MODEL SELECTION'
            loss=masked_loss(logits,y,w)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer);scaler.update()
        totals.append(loss.detach().item())
        if optimizer is None:
            # Per-chip, equal-site loss (avoid ForestGEO pixel-volume dominance)
            for i,name in enumerate(site):
                if w[i].sum()==0:continue
                l=masked_loss(logits[i:i+1],y[i:i+1],w[i:i+1])
                site_losses.setdefault(str(name),[]).append(float(l.item()))
    if optimizer is None:
        return float(np.mean([np.mean(v) for v in site_losses.values()])) if site_losses else float('inf')
    return float(np.mean(totals)) if totals else float('inf')


## 02 — One canonical overlapping inference engine (all evaluations)

In [ ]:
# GLOBAL 128 px grid, 256 px tiles, Hann blend — model probabilities, not hard masks.
# Every output core is evaluated ONCE; the outer halo supplies inference context.
from rasterio.windows import Window
from rasterio.features import rasterize
from shapely.geometry import box
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from pyproj import Transformer
import gc

EVAL_OUT=ROOT/'Outputs/v4/03_overlap_evaluation_v4_009'
EVAL_OUT.mkdir(parents=True,exist_ok=True)
V009_TILE=256
V009_STRIDE=128
assert V009_TILE==CHIP and V009_STRIDE==CHIP//2
for fold,p in CHECKPOINTS.items():
    assert p.is_file(),('Missing frozen V4.007 checkpoint',fold,p)
FOLDS=[0,1,2]
EXAMPLE_IDS={3,9,29,26}
SAVE_EXAMPLE_FIGURES=True

def load_frozen_model(fold):
    ck=torch.load(CHECKPOINTS[int(fold)],map_location='cpu',weights_only=False)
    assert ck['version']=='V4.007' and ck['fold']==int(fold) and not ck['smoke']
    assert ck.get('source_loss')=='V3_1_masked_pixel_BCE'
    m=ResNet18UNet().to(DEVICE)
    m.load_state_dict(ck['model'])
    del ck
    m.eval()
    return m

def burn(gs,ds,win):
    if not gs:return np.zeros((int(win.height),int(win.width)),dtype=bool)
    return rasterize(((g,1) for g in gs if g is not None and not g.is_empty),
        out_shape=(int(win.height),int(win.width)),
        transform=ds.window_transform(win),fill=0,dtype='uint8').astype(bool)

# Deliberately nonzero edge weight to guarantee complete support at raster limits.
hann=np.hanning(V009_TILE+2)[1:-1].astype(np.float32)
WEIGHT=np.maximum(np.outer(hann,hann),0.01).astype(np.float32)
WEIGHT/=WEIGHT.max()

def blended_window(ds,model,win):
    """Blended P(Musanga) for a core window, anchored to entire mosaic pixel grid.

    Loads only bounded core+halo RGB. No per-tile compressed-cache calls, no
    complete-orthomosaic arrays, no thresholding before probability blending.
    """
    core_r=int(win.row_off);core_c=int(win.col_off)
    h=int(win.height);w=int(win.width)
    assert h>0 and w>0 and core_r>=0 and core_c>=0
    assert core_r+h<=ds.height and core_c+w<=ds.width
    halo=V009_TILE
    top=max(0,core_r-halo);left=max(0,core_c-halo)
    bottom=min(ds.height,core_r+h+halo);right=min(ds.width,core_c+w+halo)
    full=Window(left,top,right-left,bottom-top)
    raw=ds.read([1,2,3],window=full)
    assert ds.dtypes[0]=='uint8',(ds.name,ds.dtypes)
    rgb=raw.astype(np.float32)/255.
    H,W=rgb.shape[1:]
    acc=np.zeros((H,W),np.float32)
    wt=np.zeros((H,W),np.float32)
    # Every contributing patch aligned to global origin at steps of 128 px.
    r0=((top-V009_TILE+V009_STRIDE)//V009_STRIDE)*V009_STRIDE
    c0=((left-V009_TILE+V009_STRIDE)//V009_STRIDE)*V009_STRIDE
    with torch.inference_mode():
        for gr in range(r0,bottom,V009_STRIDE):
            for gc0 in range(c0,right,V009_STRIDE):
                ar=max(top,gr);br=min(bottom,gr+V009_TILE)
                ac=max(left,gc0);bc=min(right,gc0+V009_TILE)
                if br<=ar or bc<=ac:continue
                patch=np.zeros((3,V009_TILE,V009_TILE),np.float32)
                patch[:,ar-gr:br-gr,ac-gc0:bc-gc0]=rgb[:,ar-top:br-top,ac-left:bc-left]
                xx=torch.from_numpy(patch[None]).to(DEVICE)
                with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=AMP):
                    p=torch.sigmoid(model(xx))[0,0].float().cpu().numpy()
                z=WEIGHT[ar-gr:br-gr,ac-gc0:bc-gc0]
                acc[ar-top:br-top,ac-left:bc-left]+=p[ar-gr:br-gr,ac-gc0:bc-gc0]*z
                wt[ar-top:br-top,ac-left:bc-left]+=z
    oy=core_r-top;ox=core_c-left
    sl=np.s_[oy:oy+h,ox:ox+w]
    assert np.all(wt[sl]>0),(ds.name,win,'uncovered prediction pixels')
    return (acc[sl]/wt[sl]).astype(np.float32)

def collect_metrics(p,truth,support):
    assert support.any()
    pred=p>=0.5
    tp=int((pred&truth&support).sum());fp=int((pred&~truth&support).sum())
    fn=int((~pred&truth&support).sum());tn=int((~pred&~truth&support).sum())
    n=int(support.sum())
    return dict(tp=tp,fp=fp,fn=fn,tn=tn,n=n,
        observed_cover_pct=100*(tp+fn)/n,
        predicted_cover_pct=100*(tp+fp)/n,
        predicted_probability_pct=100*float(p[support].mean()),
        bias_pp=100*(fp-fn)/n,
        pixel_F1=2*tp/(2*tp+fp+fn) if tp+fn and 2*tp+fp+fn else np.nan,
        pixel_IoU=tp/(tp+fp+fn) if tp+fn and tp+fp+fn else np.nan,
        Brier=float(np.mean((p[support]-truth[support])**2)))

def _figure(cid,mosaic,ds,win,rgb,truth,support,prob,scores):
    ds_h,ds_w=truth.shape
    step=max(1,int(np.ceil(max(ds_h,ds_w)/950)))
    sl=np.s_[::step,::step]
    view=rgb.transpose(1,2,0)[sl]
    y=truth[sl];supp=support[sl];p=prob[sl]
    cls=p>=0.5
    fig,axs=plt.subplots(1,5,figsize=(20,4.7),layout='constrained')
    axs[0].imshow(view);axs[0].set_title('RGB')
    overlay=view.copy();overlay[y&supp]=.45*overlay[y&supp]+.55*np.array([1,.1,.1])
    axs[1].imshow(np.clip(overlay,0,1));axs[1].set_title('Exhaustive reference · red')
    axs[2].imshow(view)
    a=axs[2].imshow(np.ma.masked_where(~supp,p),cmap='magma',vmin=0,vmax=1,alpha=.90)
    axs[2].set_title('Blended probability P(Musanga)')
    fig.colorbar(a,ax=axs[2],fraction=.045,pad=.02)
    overlay=view.copy();overlay[cls&supp]=.45*overlay[cls&supp]+.55*np.array([.1,1,1])
    axs[3].imshow(np.clip(overlay,0,1));axs[3].set_title('Blended threshold 0.5 · cyan')
    err=view.copy()
    for mask,color in [(cls&y&supp,[.1,1,.1]),(cls&~y&supp,[1,.1,.1]),(~cls&y&supp,[.1,.35,1])]:
        err[mask]=.3*err[mask]+.7*np.array(color)
    axs[4].imshow(np.clip(err,0,1));axs[4].set_title('TP green · FP red · FN blue')
    for ax in axs:ax.axis('off')
    fig.suptitle(f'{mosaic} | circle {cid} | observed {scores["observed_cover_pct"]:.1f}% '
                 f'→ blended {scores["predicted_cover_pct"]:.1f}% | '
                 f'bias {scores["bias_pp"]:+.1f} pp',fontsize=12)
    fig.savefig(EVAL_OUT/f'circle_{cid:02d}_blended_probability.png',dpi=160,bbox_inches='tight')
    plt.show();plt.close(fig)
print('READY — V4.009 unified 50% overlap + smooth blend, models read-only')


## 03 — FULL independent 34-circle evaluation, including probability examples

In [ ]:
# All 34 circles, full native reference support, same 128 px global grid.
# This is the PRIMARY full-circle V4.009 evaluation (not optional).
MOSAIC_BY_CIRCLE={}
for cr in CIRCLE_ROI.itertuples(index=False):
    choices=[]
    for name,meta in RGB.items():
        if meta['site']!=cr.site_key:continue
        with rasterio.open(meta['path']) as ds:
            area=cr.geometry.intersection(box(*ds.bounds)).area
        if area>0:choices.append((area,name))
    assert choices,(cr.circle_id,'Missing mosaic')
    MOSAIC_BY_CIRCLE[int(cr.circle_id)]=max(choices)[1]
CIRCLE_ROWS=[]
for fold in FOLDS:
    model=load_frozen_model(fold)
    for cr in CIRCLE_ROI.loc[CIRCLE_ROI.cv_fold.eq(fold)].itertuples(index=False):
        cid=int(cr.circle_id);mosaic=MOSAIC_BY_CIRCLE[cid]
        with rasterio.open(RGB[mosaic]['path']) as ds:
            bound=rasterio.windows.from_bounds(*cr.geometry.bounds,transform=ds.transform)
            row=max(0,int(np.floor(bound.row_off)))
            col=max(0,int(np.floor(bound.col_off)))
            bottom=min(ds.height,int(np.ceil(bound.row_off+bound.height)))
            right=min(ds.width,int(np.ceil(bound.col_off+bound.width)))
            win=Window(col,row,right-col,bottom-row)
            bb=box(*rasterio.windows.bounds(win,ds.transform))
            mask=burn([cr.geometry],ds,win)
            roi=burn(geom_hits(ROI_T,ROI_GEOMS,bb),ds,win)
            valid=ds.dataset_mask(window=win)>0
            truth=burn(geom_hits(KPOS_T,CIRCLE_POS_GEOMS,bb),ds,win)
            support=mask&roi&valid
            assert support.any(),cid
            probs=blended_window(ds,model,win)
            metrics=collect_metrics(probs,truth,support)
            metrics.update(circle_id=cid,fold=fold,site=cr.site_key,mosaic=mosaic,
                           valid_support_pct=100*support.sum()/max(1,mask.sum()),
                           centre_E=float(cr.geometry.centroid.x),
                           centre_N=float(cr.geometry.centroid.y))
            CIRCLE_ROWS.append(metrics)
            print('CIRCLE',cid,'fold',fold,'obs',round(metrics['observed_cover_pct'],2),
                  'blended',round(metrics['predicted_cover_pct'],2),
                  'F1',round(metrics['pixel_F1'],3),flush=True)
            if cid in EXAMPLE_IDS and SAVE_EXAMPLE_FIGURES:
                rgb=ds.read([1,2,3],window=win).astype('float32')/255.
                _figure(cid,mosaic,ds,win,rgb,truth,support,probs,metrics)
    del model
    torch.cuda.empty_cache();gc.collect()
CIRCLES_V009=pd.DataFrame(CIRCLE_ROWS).sort_values('circle_id')
assert len(CIRCLES_V009)==34 and CIRCLES_V009.circle_id.is_unique
CIRCLES_V009.to_csv(EVAL_OUT/'V4_009_full34_blended_circles.csv',index=False)
def coverage_summary(g):
    yy=g.observed_cover_pct.to_numpy(float);pp=g.predicted_cover_pct.to_numpy(float)
    return pd.Series(dict(n=len(g),R2=r2_score(yy,pp) if len(g)>1 else np.nan,
        MAE_pp=np.mean(np.abs(pp-yy)),RMSE_pp=np.sqrt(np.mean((pp-yy)**2)),
        bias_pp=np.mean(pp-yy),mean_pixel_F1=g.pixel_F1.mean(),
        mean_pixel_IoU=g.pixel_IoU.mean(),mean_Brier=g.Brier.mean()))
CIRCLE_METRICS=pd.concat([
    coverage_summary(CIRCLES_V009).to_frame().T.assign(site='ALL'),
    CIRCLES_V009.groupby('site').apply(coverage_summary,include_groups=False).reset_index()
],ignore_index=True)
CIRCLE_METRICS.to_csv(EVAL_OUT/'V4_009_circle_metrics_by_site.csv',index=False)
display(CIRCLE_METRICS.round(3))


## 04 — FULL 78-plot fold manifest evaluation (76 expected RGB-evaluable)

In [ ]:
# Plot evaluation uses every unmodified outer 'val' manifest window (no sampling).
# Buffered overlapping inference is run only over selected validation windows;
# label masks remain strictly inside each valid held-out plot and ROI.
PLOT_ROWS=[]
for fold in FOLDS:
    model=load_frozen_model(fold)
    state={}
    for pl in PLOTS.loc[PLOTS.cv_fold.eq(fold)].itertuples(index=False):
        state[str(pl.Plot_name)]=dict(tp=0,fp=0,fn=0,tn=0,n=0,sum_prob=0.,sum_brier=0.,
                                     site=pl.site_key)
    rows=SPLITS.loc[(SPLITS.cv_fold==fold)&SPLITS.split.eq('val')]
    for mosaic,g in rows.groupby('mosaic',sort=True):
        with rasterio.open(RGB[mosaic]['path']) as ds:
            for z in g.itertuples(index=False):
                # Native 512px manifest window, evaluated once, probabilities blended
                win=Window(int(z.col),int(z.row),int(z.width),int(z.height))
                bb=box(*rasterio.windows.bounds(win,ds.transform))
                hits=[PLOTS.iloc[int(i)] for i in P_T.query(bb,predicate='intersects')
                      if PLOTS.iloc[int(i)].site_key==z.site
                      and int(PLOTS.iloc[int(i)].cv_fold)==fold]
                if not hits:continue
                roi=burn(geom_hits(ROI_T,ROI_GEOMS,bb),ds,win)
                valid=ds.dataset_mask(window=win)>0
                # Source reference is identical to V4.007: plot polygons and positive crowns.
                pos=burn(geom_hits(CP_T,CROWN_POS,bb),ds,win)
                prob=blended_window(ds,model,win)
                for pl in hits:
                    s=burn([pl.geometry],ds,win)&roi&valid
                    if not s.any():continue
                    p=prob[s];y=pos[s]
                    pred=p>=.5
                    name=str(pl.Plot_name);d=state[name]
                    d['tp']+=int((pred&y).sum());d['fp']+=int((pred&~y).sum())
                    d['fn']+=int((~pred&y).sum());d['tn']+=int((~pred&~y).sum())
                    d['n']+=len(y);d['sum_prob']+=float(p.sum())
                    d['sum_brier']+=float(((p-y)**2).sum())
    for name,d in state.items():
        n=d['n'];pos=d['tp']+d['fn'];pred=d['tp']+d['fp']
        PLOT_ROWS.append(dict(fold=fold,site=d['site'],Plot_name=name,
           n_valid=n,observed_cover_pct=100*pos/n if n else np.nan,
           predicted_cover_pct=100*pred/n if n else np.nan,
           predicted_probability_pct=100*d['sum_prob']/n if n else np.nan,
           bias_pp=100*(pred-pos)/n if n else np.nan,
           abs_error_pp=100*abs(pred-pos)/n if n else np.nan,
           tp=d['tp'],fp=d['fp'],fn=d['fn'],tn=d['tn'],
           pixel_F1=2*d['tp']/(2*d['tp']+d['fp']+d['fn']) if pos and 2*d['tp']+d['fp']+d['fn'] else np.nan,
           pixel_IoU=d['tp']/(d['tp']+d['fp']+d['fn']) if pos and d['tp']+d['fp']+d['fn'] else np.nan,
           Brier=d['sum_brier']/n if n else np.nan))
    print('COMPLETE PLOT FOLD',fold,'manifest windows',len(rows),flush=True)
    del model
    torch.cuda.empty_cache();gc.collect()
PLOTS_V009=pd.DataFrame(PLOT_ROWS)
assert len(PLOTS_V009)==78 and PLOTS_V009.Plot_name.nunique()==78
assert int((PLOTS_V009.n_valid>0).sum())==76,('Unexpected plot coverage; investigate manifest/ROI')
PLOTS_V009.to_csv(EVAL_OUT/'V4_009_full78_plots_76evaluable_blended.csv',index=False)
PLOT_METRICS=pd.concat([
    coverage_summary(PLOTS_V009[PLOTS_V009.n_valid>0]).to_frame().T.assign(site='ALL'),
    PLOTS_V009[PLOTS_V009.n_valid>0].groupby('site').apply(coverage_summary,include_groups=False).reset_index()
],ignore_index=True)
PLOT_METRICS.to_csv(EVAL_OUT/'V4_009_plot_metrics_by_site.csv',index=False)
display(PLOT_METRICS.round(3))
print('V4.009 COMPLETE — 34 circles and 76 plots using SAME 50% overlap inference')


## 05 — Paired before/after summary plots and reference comparisons

In [ ]:
# Compare against original V4.007 evaluation summaries, where available.
OLD_DIR=ROOT/'Outputs/v4/02_rgb_unet_field78_v4_007_full'
old34=OLD_DIR/'performance_report/circles_full_v3style/02_v4_full34_circles_by_circle.csv'
old78=OLD_DIR/'02_rgb_heldout_per_plot.csv'
if old34.is_file():
    old=pd.read_csv(old34)
    paired=CIRCLES_V009.merge(old[['circle_id','predicted_fraction']],
                             on='circle_id',validate='one_to_one')
    paired['old_abs_err']=abs(paired.predicted_fraction-paired.observed_cover_pct)
    paired['new_abs_err']=abs(paired.predicted_cover_pct-paired.observed_cover_pct)
    paired.to_csv(EVAL_OUT/'V4_009_circle_paired_original_vs_blended.csv',index=False)
    print('34 circles: MAE old',round(paired.old_abs_err.mean(),3),
          '→ blended',round(paired.new_abs_err.mean(),3))
if old78.is_file():
    old=pd.read_csv(old78)
    paired=PLOTS_V009.merge(old[['Plot_name','predicted_cover_pct']],
                            on='Plot_name',suffixes=('_new','_old'),validate='one_to_one')
    paired.to_csv(EVAL_OUT/'V4_009_plot_paired_original_vs_blended.csv',index=False)
    print('76 evaluable plots: paired output saved.')
fig,ax=plt.subplots(1,2,figsize=(12,5.4),layout='constrained')
palette={'Baego':'#3168B4','MOS':'#DE8E32','Yangambi':'#25856A','Yoko':'#9A5080'}
for i,(frame,title) in enumerate([(CIRCLES_V009,'34 exhaustive circles'),
                                   (PLOTS_V009[PLOTS_V009.n_valid>0],'76 exhaustive plots')]):
    for site,d in frame.groupby('site'):
        ax[i].scatter(d.observed_cover_pct,d.predicted_cover_pct,
                      label=site,color=palette[site],alpha=.8,s=36)
    ax[i].plot([0,100],[0,100],'--',color='.5')
    ax[i].set(xlim=(-2,102),ylim=(-2,102),aspect='equal',
       xlabel='Reference Musanga cover (%)',ylabel='Blended predicted cover (%)',
       title=title)
    ax[i].legend(frameon=False,fontsize=8)
fig.savefig(EVAL_OUT/'V4_009_full_blended_scatterplots.png',dpi=200,bbox_inches='tight')
plt.show();plt.close(fig)


## 06 — Read-only inventory of existing structural / spectral inputs

In [ ]:
# This does not claim all sources are co-registered: audit the actual filenames,
# CRSs, GSD, bounds and sample acquisition date prior to any combined model.
print('NB00 manifest fields:',sorted(NB00))
for k,v in NB00.items():
    if any(term in str(k).lower() for term in ['dsm','dtm','chm','ms','multi','spect','predict','orth','ortho','lidar']):
        print(k,':',str(v)[:450])
candidate_dirs=[DATA/'Ortho_MS',DATA/'Ortho_Multispectral',DATA/'DSM',
                DATA/'DTM',DATA/'CHM',DATA/'Ortho_RGB']
inventory=[]
for folder in candidate_dirs:
    if not folder.is_dir():continue
    for path in sorted(folder.rglob('*')):
        if path.suffix.lower() not in ('.tif','.tiff','.vrt','.jp2'):continue
        if 'Ortho_RGB' in str(path) and len(inventory)>60:continue
        try:
            with rasterio.open(path) as ds:
                inventory.append(dict(path=str(path.relative_to(DATA)),
                  bands=ds.count,crs=str(ds.crs),gsd_x=abs(ds.transform.a),
                  gsd_y=abs(ds.transform.e),width=ds.width,height=ds.height,
                  bounds=str(tuple(round(x,1) for x in ds.bounds))))
        except Exception as e:
            inventory.append(dict(path=str(path.relative_to(DATA)),error=str(e)))
inventory=pd.DataFrame(inventory)
inventory.to_csv(EVAL_OUT/'V4_009_predictor_file_inventory.csv',index=False)
display(inventory)
print('Inventory is metadata only; exact spatial alignment and image provenance still require QA.')
